# The Craft of Data-Driven Storytelling

**Team Members:** Name · Name · Name · Name

**Project Title:** *Your project title here*

---

## Getting Started

- Replace the team member names above with all contributors
- Replace *Your project title here* with your actual project title
- One team member hosts this notebook and shares it with the full team for editing
- When finished, the host changes sharing to **Anyone with the link can view** and submits to Canvas
- Every team member's name must appear above — only one submission per team

---

## How to Use This Notebook

This notebook is designed to be accessible to all users, including those navigating with a **screen reader**.

### Screen Reader Navigation Guide

Before each code cell you will find a navigation landmark with two options:

- **Skip this code cell** — Jumps past the code to the explanation.
- **Go back and read the code cell** — Returns you to the top of the code section.

**Tips:** Press **H** to jump between headings, **K** or **Tab** to move between links, **D** to jump between landmark regions.

---

## What This Notebook Is

This is a course in craft. Not in tools. Not in models.

Data science programs teach students to find patterns. What they rarely teach is how to make other people see them.

A model that achieves 94% accuracy means nothing to a decision-maker who cannot understand what 94% accuracy implies for their business. A finding that would change an organization's direction goes nowhere if it is buried in a table that takes three minutes to decode.

The gap between insight and impact is a communication problem. This notebook is about closing that gap.

## What You Will Produce

By the end of this notebook your team will have built a **data story** about your final project — a structured, narrative document that makes your findings legible to any reader, not just a data scientist. Each section of the notebook teaches one storytelling principle and immediately asks your team to apply it to your own data.

---

## Notebook Roadmap

| Principle | Core Question |
| :--- | :--- |
| **1. The Three Pillars** | What makes a data story, and what is each element's job? |
| **2. Find the Story First** | What does your data actually say — and why does it matter? |
| **3. Know Your Reader** | Who are you talking to, and what do they need to believe? |
| **4. One Claim Per Chart** | What is the single argument each visualization makes? |
| **5. Declutter** | What can you remove without losing the point? |
| **6. Color as Emphasis** | Where do you want the eye to go first? |
| **7. Annotate the Insight** | Have you written the story on the chart, or just beside it? |
| **Team Data Story** | Apply all seven principles to your own project data |

---

<a name="read-code-1"></a>

**Setup Cell — Install and Import All Libraries**

This cell installs Plotly (for interactive, publication-quality charts) and imports everything needed throughout the notebook. Plotly Express (`px`) will be the primary visualization library — it produces the kind of clean, interactive charts appropriate for data stories. Matplotlib will be used for the makeover examples, where fine-grained control over every element is necessary.

<nav aria-label="Code cell 1 navigation">
<a href="#skip-code-1" aria-label="Skip code cell 1 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly', 'kaleido', '-q'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Load the Gapminder dataset — the teaching vehicle for this notebook
gapminder = px.data.gapminder()

print(f'Gapminder dataset loaded: {gapminder.shape[0]:,} rows, {gapminder.shape[1]} columns')
print(f'Countries: {gapminder.country.nunique()}')
print(f'Years: {gapminder.year.min()} to {gapminder.year.max()}')
print(f'Columns: {list(gapminder.columns)}')
gapminder.head()

<a name="skip-code-1"></a>

**Expected output:** 1,704 rows covering 142 countries from 1952 to 2007 with life expectancy, population, and GDP per capita per country per year.

This dataset was made famous by the Swedish physician and statistician Hans Rosling, whose 2006 TED talk 'The best stats you've ever seen' used it to overturn decades of conventional wisdom about global development — using animation, narrative, and data that had been publicly available for years but that no one had thought to visualize this way. The insight was not in the data. It was in the story.

<nav aria-label="Return navigation for code cell 1">
<a href="#read-code-1" aria-label="Go back and read code cell 1">&#8617; Go back and read the code cell</a>
</nav>

---
# Principle 1: The Three Pillars

Every data story is built on three elements. Remove any one of them and the story fails.

## Data

Data is the evidence. It is what makes your story true rather than merely interesting. But data without interpretation is just numbers — it proves nothing to someone who has not already done the analysis themselves. A table of 1,704 rows tells no one anything.

## Narrative

Narrative is the meaning. It is the answer to the question every reader is silently asking: *So what?* Narrative explains why the numbers matter, what they imply, and what should change because of them. Without narrative, even a beautiful chart is just decoration.

## Visual

Visual is the delivery. A well-designed chart communicates the finding in seconds and makes it memorable. Research on narrative transportation consistently shows that people who see information presented visually with a story retain it longer and are more likely to act on it than those who read the equivalent table.

The three pillars are not interchangeable. Data tells you what is true. Narrative tells you why it matters. Visual makes both accessible.

## The Problem with Tables

The cell below shows the same information in two forms: a data table and a chart. Both are accurate. Only one tells a story.

---

<a name="read-code-2"></a>

**Cell 2 — The Same Data, Two Forms**

This cell produces two outputs for the same slice of data: a raw table showing life expectancy across continents in 2007, and a bar chart showing the same numbers. The question to hold in your mind as you look at both: which one makes you feel something? Which one are you more likely to remember tomorrow?

<nav aria-label="Code cell 2 navigation">
<a href="#skip-code-2" aria-label="Skip code cell 2 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
gap_2007 = gapminder[gapminder.year == 2007].copy()
continent_summary = (
    gap_2007.groupby('continent')
    .agg(avg_life_exp=('lifeExp','mean'), avg_gdp=('gdpPercap','mean'), n_countries=('country','count'))
    .round(1).reset_index()
    .sort_values('avg_life_exp', ascending=False)
)

print('=== The Table (Accurate. Boring.) ===')
print(continent_summary.to_string(index=False))

# The Chart
fig = px.bar(
    continent_summary,
    x='continent', y='avg_life_exp',
    color='avg_life_exp',
    color_continuous_scale='Blues',
    labels={'avg_life_exp': 'Average Life Expectancy (years)', 'continent': ''},
    title='Average Life Expectancy by Continent, 2007',
    text='avg_life_exp'
)
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.update_layout(
    coloraxis_showscale=False,
    plot_bgcolor='white', paper_bgcolor='white',
    font_family='Arial', font_size=13,
    title_font_size=16,
    showlegend=False
)
fig.update_yaxes(range=[0, 85], gridcolor='#f0f0f0')
fig.show()

<a name="skip-code-2"></a>

**The table is complete. The chart is a story.**

The table requires the reader to do work: scan down the column, compare numbers mentally, calculate the gap between highest and lowest. The chart makes that comparison instant — the eye does the work before the brain even engages.

But here is the deeper point: this chart is still not a story. It is a visualization. A story would tell you *why* Oceania is highest, *what happened* to make Africa lowest, and *what this means* for anyone reading it. The visual is just the first step. Narrative is what makes it matter.

<nav aria-label="Return navigation for code cell 2">
<a href="#read-code-2" aria-label="Go back and read code cell 2">&#8617; Go back and read the code cell</a>
</nav>

## ✏️ Team Reflection 1

**Edit this Markdown cell and answer the following question as a team.**

Look at the chart above. Write two to three sentences that turn it into a story. Do not describe what the chart shows — interpret it. Answer the 'so what' question: why does this pattern exist, and why should a reader care?

*Your answer here.*

---

---
# Principle 2: Find the Story Before You Visualize

The most common mistake in data communication is opening a visualization tool before knowing what you want to say.

When you visualize without a story in mind, you produce exploratory charts — charts that help *you* understand the data. These are essential for analysis, but they are not for your audience. A chart built for exploration shows everything. A chart built for communication shows one thing clearly.

## The 'So What?' Test

Before building any visualization, write down the answer to this question in one sentence:

> *What do I want my reader to believe, understand, or do differently after seeing this chart?*

If you cannot answer this in one sentence, you are not ready to visualize.

Possible answers:
- *'I want the reader to understand that life expectancy in Africa has improved by 15 years since 1960.'*
- *'I want the reader to see that the gap between rich and poor countries has narrowed over 50 years.'*
- *'I want the reader to recognize that high GDP does not always mean high life expectancy.'*

Each of these answers implies a different chart. The first wants a time series for Africa. The second wants a comparison over time across income groups. The third wants a scatter plot showing the relationship — or lack thereof.

## The Narrative Arc

Every data story worth telling has a shape:

| Stage | Question It Answers | In Plain Terms |
| :--- | :--- | :--- |
| **Hook** | Why should I keep reading? | A surprising, counterintuitive, or urgent finding |
| **Context** | What was the situation before? | The world as it was |
| **Tension** | What did the data reveal? | The unexpected finding |
| **Resolution** | What changed or what does this mean? | The insight |

Hans Rosling's arc: *You think the world is still divided into rich/healthy and poor/sick. (Context) But look at what happened between 1952 and 2007. (Tension) Most of the world has caught up — the old categories no longer describe reality. (Resolution)*

---

<a name="read-code-3"></a>

**Cell 3 — Finding the Story: Three Different Stories from the Same Data**

This cell produces three charts from the same Gapminder dataset, each built around a different 'so what' question. Notice how the choice of story comes first, and the visualization follows — not the reverse. Each chart is an argument, not a display.

<nav aria-label="Code cell 3 navigation">
<a href="#skip-code-3" aria-label="Skip code cell 3 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '"Africa has caught up more than\nyou think"',
        '"Wealth and health are strongly\ncorrelated — but not perfectly"',
        '"The gap between best and worst\nhas narrowed dramatically"
    ],
    horizontal_spacing=0.08
)

# Story 1: Africa life expectancy over time
africa = gapminder[gapminder.continent == 'Africa'].groupby('year')['lifeExp'].mean().reset_index()
fig.add_trace(go.Scatter(
    x=africa.year, y=africa.lifeExp,
    mode='lines+markers', line=dict(color='#e67e22', width=3),
    showlegend=False, name='Africa avg'
), row=1, col=1)
fig.add_annotation(x=2007, y=africa[africa.year==2007]['lifeExp'].values[0],
    text=f"{africa[africa.year==2007]['lifeExp'].values[0]:.1f} yrs",
    showarrow=True, arrowhead=2, ax=30, ay=-20,
    font=dict(size=11, color='#e67e22'), row=1, col=1)

# Story 2: GDP vs life expectancy scatter (2007)
g2007 = gapminder[gapminder.year == 2007]
fig.add_trace(go.Scatter(
    x=g2007.gdpPercap, y=g2007.lifeExp,
    mode='markers',
    marker=dict(
        size=g2007['pop'] / 5e6 + 4,
        color=g2007.continent.map({'Africa':'#e74c3c','Americas':'#3498db',
                                   'Asia':'#f39c12','Europe':'#27ae60','Oceania':'#9b59b6'}),
        opacity=0.7, line=dict(width=0.5, color='white')
    ),
    showlegend=False
), row=1, col=2)

# Story 3: Best vs Worst life expectancy over time
annual = gapminder.groupby('year')['lifeExp'].agg(['max','min']).reset_index()
fig.add_trace(go.Scatter(
    x=annual.year, y=annual['max'], mode='lines',
    line=dict(color='#27ae60', width=2), name='Highest country',
    showlegend=False
), row=1, col=3)
fig.add_trace(go.Scatter(
    x=annual.year, y=annual['min'], mode='lines',
    line=dict(color='#e74c3c', width=2), name='Lowest country',
    fill='tonexty', fillcolor='rgba(200,200,200,0.2)',
    showlegend=False
), row=1, col=3)

fig.update_layout(
    height=380, paper_bgcolor='white', plot_bgcolor='white',
    font_family='Arial', font_size=11,
    title_text='Three Different Stories — Same Dataset, Different "So What" Questions',
    title_font_size=14
)
fig.update_xaxes(gridcolor='#f5f5f5', linecolor='#cccccc')
fig.update_yaxes(gridcolor='#f5f5f5', linecolor='#cccccc')
fig.show()

<a name="skip-code-3"></a>

**Each chart is an argument.** The first says: Africa has made real progress — your mental model of the continent may be outdated. The second says: money buys health, but not infinitely — notice the flattening at high GDP. The third says: the distance between the best and worst countries has shrunk. None of these messages were in the data until someone asked a question.

The data did not decide what to say. The analyst did.

<nav aria-label="Return navigation for code cell 3">
<a href="#read-code-3" aria-label="Go back and read code cell 3">&#8617; Go back and read the code cell</a>
</nav>

## ✏️ Team Task 2: Write Your 'So What' Sentences

Before your team builds any visualization for your own project in the Team Data Story section, you must answer these three questions. Write them in this Markdown cell now. You will return to them when building your story.

**1. What is the single most important thing your data reveals?**

*Write one sentence.*

**2. Who would be most surprised by this finding — and why?**

*Write one to two sentences.*

**3. What should someone do differently after seeing your story?**

*Write one sentence. This is your Call to Action.*

---
**YOUR ANSWERS HERE**

---

---
# Principle 3: Know Your Reader

The same insight requires a completely different story depending on who is reading it.

Consider the finding: *'Our model achieves 0.94 AUC on the test set, compared to 0.79 for the baseline logistic regression.'*

| Reader | What They Need to Hear | How to Frame It |
| :--- | :--- | :--- |
| **Data scientist colleague** | Technical detail — how did you evaluate? What's the confidence interval? | Give them the ROC curve and the methodology |
| **Product manager** | Business impact — what does this mean for the user experience? | 'Our model catches 18% more at-risk users before they churn' |
| **CFO** | Financial impact — what does this cost and what does it save? | 'Reducing false negatives by 18% recovers approximately $2.4M in annual retention' |
| **Industry expert (seminar)** | Credibility — is this real? Is the team rigorous? | Show the method, then the business translation — in that order |

The number 0.94 AUC appears in none of the last three framings. That does not mean it is unimportant — it means it is supporting evidence, not the headline. The headline is what the number *implies*.

## The Reader's Three Questions

Every reader, regardless of background, is silently asking:

1. **Why should I care?** (Make this clear in the first 30 seconds)
2. **Can I trust this?** (Evidence, methodology, honest acknowledgment of limitations)
3. **What am I supposed to do?** (The call to action)

If your story does not answer all three, it is incomplete.

---

<a name="read-code-4"></a>

**Cell 4 — The Same Finding, Three Audiences**

This cell demonstrates how one Gapminder finding — the relationship between income and child mortality — can be framed three different ways for three different readers. The data does not change. The story does.

<nav aria-label="Code cell 4 navigation">
<a href="#skip-code-4" aria-label="Skip code cell 4 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# One finding: income and child mortality (inverse of lifeExp as proxy)
g2007 = gapminder[gapminder.year == 2007].copy()
g2007['child_mortality_proxy'] = 100 - g2007['lifeExp']  # Simplified proxy for illustration

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('white')

# ── Frame 1: For a Data Scientist ────────────────────────────────────────────
ax1 = axes[0]
colors_continent = {
    'Africa':'#e74c3c','Americas':'#3498db','Asia':'#f39c12',
    'Europe':'#27ae60','Oceania':'#9b59b6'
}
for continent, grp in g2007.groupby('continent'):
    ax1.scatter(np.log10(grp.gdpPercap), grp.lifeExp,
                label=continent, color=colors_continent[continent],
                alpha=0.65, s=grp['pop']/1e6, edgecolors='none')
ax1.set_xlabel('GDP per capita (log₁₀ scale)', fontsize=10)
ax1.set_ylabel('Life Expectancy (years)', fontsize=10)
ax1.set_title('For the Data Scientist:\n"Log-linear relationship,\nR² ≈ 0.64 (2007 cross-section)"',
              fontsize=10, fontweight='bold', pad=8)
ax1.legend(fontsize=7, loc='lower right')
ax1.grid(alpha=0.2)
ax1.tick_params(labelsize=8)

# ── Frame 2: For the Policy Maker ────────────────────────────────────────────
ax2 = axes[1]
income_groups = pd.qcut(g2007.gdpPercap, q=4, labels=['Lowest 25%','Lower-mid','Upper-mid','Highest 25%'])
g2007['income_group'] = income_groups
means = g2007.groupby('income_group', observed=True)['lifeExp'].mean()
bars = ax2.bar(means.index, means.values,
               color=['#e74c3c','#e67e22','#f1c40f','#27ae60'],
               edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, means.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}\nyrs', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.set_title('For the Policy Maker:\n"People in the poorest countries\nlive 19 fewer years on average"',
              fontsize=10, fontweight='bold', pad=8)
ax2.set_ylabel('Average Life Expectancy', fontsize=10)
ax2.set_ylim(0, 88)
ax2.tick_params(axis='x', labelsize=8)
ax2.spines[['top','right']].set_visible(False)
ax2.grid(axis='y', alpha=0.2)

# ── Frame 3: For the General Audience ────────────────────────────────────────
ax3 = axes[2]
low  = g2007[g2007.gdpPercap < g2007.gdpPercap.quantile(0.25)]['lifeExp']
high = g2007[g2007.gdpPercap > g2007.gdpPercap.quantile(0.75)]['lifeExp']
ax3.barh(['Wealthiest\n25% of countries','Poorest\n25% of countries'],
         [high.mean(), low.mean()],
         color=['#27ae60','#e74c3c'], edgecolor='white', height=0.5)
for i, (val, color) in enumerate([(high.mean(),'#27ae60'),(low.mean(),'#e74c3c')]):
    ax3.text(val + 0.5, i, f'{val:.0f} years', va='center', fontsize=12, fontweight='bold', color=color)
gap = high.mean() - low.mean()
ax3.set_xlim(0, 88)
ax3.set_title(f'For the General Audience:\n"Where you are born determines\nhow long you live — by {gap:.0f} years"',
              fontsize=10, fontweight='bold', pad=8)
ax3.spines[['top','right','left']].set_visible(False)
ax3.tick_params(axis='y', labelsize=10)
ax3.xaxis.set_visible(False)

plt.suptitle('Same Data. Same Finding. Three Readers. Three Stories.',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('three_audiences.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-4"></a>

**The finding in each chart is the same: wealthy countries have longer life expectancy.** What changes is what the reader is expected to do with that information.

The data scientist wants to evaluate the model fit. The policy maker wants to see the magnitude of the disparity in actionable terms. The general audience needs the gap made concrete and personal — 19 years is abstract; *'where you are born determines how long you live'* is not.

Notice that the third chart uses the fewest numbers and the simplest design. This is not dumbing down — it is precision. A chart for a non-technical audience requires *more* thoughtful design, not less.

<nav aria-label="Return navigation for code cell 4">
<a href="#read-code-4" aria-label="Go back and read code cell 4">&#8617; Go back and read the code cell</a>
</nav>

## ✏️ Team Task 3: Define Your Reader

For the industry expert audience at your seminar, answer the following. Be specific — generic answers produce generic stories.

**Who specifically is reading your data story?**
*(Job title, industry, what they care about, what decisions they make)*

*Your answer.*

**What do they already know — and what will surprise them?**

*Your answer.*

**What do you need them to believe by the end?**
*(One sentence — the belief that will lead to the action you want)*

*Your answer.*

---

---
# Principle 4: One Claim Per Chart

Every chart you include in a data story should be doing one job: making one argument.

When a chart tries to make multiple points simultaneously, readers do not absorb both — they absorb neither. The eye scans for the most prominent element, decides it understands, and moves on.

## Choosing the Right Chart Type

Chart type is not an aesthetic choice. It is a rhetorical one. Each chart type makes a different implicit claim.

| If you want to show... | Use... | Not... |
| :--- | :--- | :--- |
| Change over time | Line chart | Bar chart, pie chart |
| Comparison across categories | Bar chart (horizontal) | Pie chart, 3D chart |
| Relationship between two variables | Scatter plot | Bar chart, line chart |
| Distribution of one variable | Histogram, box plot | Pie chart, bar chart |
| Part of a whole | Stacked bar, waffle chart | 3D pie chart |
| Geographic patterns | Choropleth map | Bar chart |

## The Pie Chart Problem

Pie charts are the most abused chart type in professional communication. The human eye is remarkably poor at comparing areas and angles — which is exactly what a pie chart requires. A bar chart representing the same data allows instant comparison because the eye is very good at comparing lengths.

There is one situation where a pie chart is appropriate: when you have two categories and you want to show that one is overwhelmingly dominant. In almost every other situation, a bar chart is more honest and more readable.

---

<a name="read-code-5"></a>

**Cell 5 — Chart Type as Argument: The Same Data, Wrong and Right**

This cell produces two pairs of charts using the same Gapminder data: a pie chart versus a bar chart, and a cluttered multi-line chart versus a focused comparison. The wrong choice is not wrong because it is ugly — it is wrong because it makes the argument harder to see.

<nav aria-label="Code cell 5 navigation">
<a href="#skip-code-5" aria-label="Skip code cell 5 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('white')

g2007 = gapminder[gapminder.year == 2007]
cont_pop = g2007.groupby('continent')['pop'].sum().reset_index().sort_values('pop', ascending=False)

# ── Wrong: Pie chart ─────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.pie(cont_pop['pop'],
       labels=cont_pop['continent'],
       autopct='%1.1f%%',
       colors=['#3498db','#e74c3c','#f39c12','#27ae60','#9b59b6'],
       startangle=90, pctdistance=0.7)
ax.set_title('World Population by Continent, 2007\n❌ Pie chart — compare Asia and Africa',
             fontsize=11, color='#c0392b', pad=8)

# ── Right: Horizontal bar chart ──────────────────────────────────────────────
ax = axes[0, 1]
bars = ax.barh(cont_pop['continent'][::-1],
               cont_pop['pop'][::-1] / 1e9,
               color='#3498db', alpha=0.85, edgecolor='white')
for bar, val in zip(bars, (cont_pop['pop'][::-1] / 1e9)):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}B', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Population (billions)', fontsize=10)
ax.set_title('World Population by Continent, 2007\n✓ Bar chart — Asia vs Africa is instantly clear',
             fontsize=11, color='#27ae60', pad=8)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='x', alpha=0.2)
ax.set_xlim(0, 5)

# ── Wrong: All continents, one chart, no focus ───────────────────────────────
ax = axes[1, 0]
for cont in gapminder.continent.unique():
    data = gapminder[gapminder.continent == cont].groupby('year')['lifeExp'].mean()
    ax.plot(data.index, data.values, marker='o', markersize=3, linewidth=1.5, label=cont)
ax.set_xlabel('Year', fontsize=10); ax.set_ylabel('Life Expectancy', fontsize=10)
ax.set_title('Life Expectancy Over Time\n❌ Five equally weighted lines — where is the story?',
             fontsize=11, color='#c0392b', pad=8)
ax.legend(fontsize=8); ax.grid(alpha=0.2)

# ── Right: One story — Africa's progress, others as context ──────────────────
ax = axes[1, 1]
for cont in gapminder.continent.unique():
    data = gapminder[gapminder.continent == cont].groupby('year')['lifeExp'].mean()
    if cont == 'Africa':
        ax.plot(data.index, data.values, color='#e67e22', linewidth=3,
                marker='o', markersize=4, zorder=5, label='Africa')
        ax.annotate(f"{data[2007]:.1f} yrs",
                    xy=(2007, data[2007]), xytext=(1999, data[2007]+3),
                    fontsize=10, color='#e67e22', fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='#e67e22', lw=1.5))
    else:
        ax.plot(data.index, data.values, color='#cccccc', linewidth=1, alpha=0.6)
ax.set_xlabel('Year', fontsize=10); ax.set_ylabel('Life Expectancy', fontsize=10)
ax.set_title('Africa Has Gained 20 Years of Life Expectancy\n✓ One story, one color — others provide context',
             fontsize=11, color='#27ae60', pad=8)
ax.legend(fontsize=9); ax.grid(alpha=0.15)
ax.spines[['top','right']].set_visible(False)

plt.suptitle('Chart Type Is a Rhetorical Choice', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('chart_type_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-5"></a>

**The right chart is the one that makes your argument in the fewest seconds.**

In the top row: the pie chart makes you work to compare Asia and Africa — you have to read the percentages, hold them in working memory, and compute the difference. The bar chart shows it in a single glance.

In the bottom row: the left chart is technically complete — all five continents are present. But it has no argument. The right chart has exactly one: *Africa has gained 20 years of life expectancy.* Everything else is context. The gray lines say: compared to the rest of the world, Africa started lower but has been catching up.

<nav aria-label="Return navigation for code cell 5">
<a href="#read-code-5" aria-label="Go back and read code cell 5">&#8617; Go back and read the code cell</a>
</nav>

---
# Principle 5: Declutter

Edward Tufte, the statistician and data visualization pioneer, introduced the concept of the **data-ink ratio**: the proportion of a chart's ink (or pixels) that represent actual data versus the proportion that serves only decorative purposes.

> *'Erase non-data-ink, within reason. Erase redundant data-ink, within reason.'* — Edward Tufte

The goal is not minimalism for its own sake. The goal is removing anything that competes with your reader's attention without adding information.

## What to Remove

Work through this checklist before finalizing any visualization:

| Element | Remove if... |
| :--- | :--- |
| Gridlines | They are decorative, not necessary for reading the values |
| Border/frame around the chart | Almost always unnecessary |
| 3D effects | They distort perception of values — remove always |
| Legend | You can label the data directly instead |
| Tick marks | Reduce to only what is needed to read the scale |
| Redundant axis labels | If units are in the title, remove from axis |
| Default colors | If color is not doing semantic work, simplify to one color |
| Data labels AND axis | Rarely need both — choose one |

The principle: **if removing an element does not change what the chart communicates, remove it.**

---

<a name="read-code-6"></a>

**Cell 6 — The Chart Makeover: Before and After Decluttering**

This cell produces the same chart twice: once with all of matplotlib's default settings (which include many elements that add visual noise without adding information), and once after applying systematic decluttering. The data and the finding are identical. The readability is not.

<nav aria-label="Code cell 6 navigation">
<a href="#skip-code-6" aria-label="Skip code cell 6 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
asia_2007 = gapminder[(gapminder.continent == 'Asia') & (gapminder.year == 2007)]
top_asia  = asia_2007.nlargest(10, 'lifeExp').sort_values('lifeExp')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')

# ── BEFORE: Default matplotlib (cluttered) ───────────────────────────────────
ax = axes[0]
ax.barh(top_asia['country'], top_asia['lifeExp'],
        color='steelblue', edgecolor='black', linewidth=0.7)
ax.set_xlabel('Life Expectancy (years)')
ax.set_title('Life Expectancy — Top 10 Asian Countries (2007)\n❌ Before Decluttering', pad=10)
ax.grid(True, linestyle='--', alpha=0.7)  # Gridlines on both axes
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.set_xlim(70, 85)
ax.legend(['Life Expectancy (years)'])  # Unnecessary legend

# ── AFTER: Decluttered ───────────────────────────────────────────────────────
ax = axes[1]
bars = ax.barh(top_asia['country'], top_asia['lifeExp'],
               color='#7fb3d3', edgecolor='none')  # Softer color, no border

# Direct labels instead of axis
for bar, val in zip(bars, top_asia['lifeExp']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', ha='left', fontsize=10)

# Highlight the story — Japan leads
bars[-1].set_color('#1a5276')  # Japan is last (longest) after sorting
ax.text(top_asia['lifeExp'].max() + 0.1,
        len(top_asia) - 1, f'{top_asia["lifeExp"].max():.1f}',
        va='center', ha='left', fontsize=10, fontweight='bold', color='#1a5276')

ax.set_title('Japan Leads Asian Life Expectancy at 82.6 Years\n✓ After Decluttering', pad=10,
             fontsize=11, fontweight='bold')
ax.set_xlim(70, 87)
ax.set_xlabel('')       # Remove redundant axis label — values are directly labeled
ax.xaxis.set_visible(False)   # Remove x-axis entirely — direct labels replace it
ax.spines[['top','right','left','bottom']].set_visible(False)  # Remove all borders
ax.tick_params(axis='y', labelsize=10)
ax.grid(False)

plt.tight_layout()
plt.savefig('declutter_makeover.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-6"></a>

**The before chart is accurate. The after chart is readable.**

What was removed: gridlines, chart border, legend, x-axis labels, axis ticks. What was added: direct data labels, a highlighted bar for the leading country, and a title that states the finding rather than describes the chart.

Notice that the after chart has a title that makes a claim: *'Japan leads Asian life expectancy at 82.6 years.'* The before chart's title describes what is in the chart: *'Life expectancy — Top 10 Asian Countries.'* One tells the reader what to think. The other makes the reader figure it out.

<nav aria-label="Return navigation for code cell 6">
<a href="#read-code-6" aria-label="Go back and read code cell 6">&#8617; Go back and read the code cell</a>
</nav>

---
# Principle 6: Color as Emphasis

Color is the most powerful visual tool in data communication — and the most consistently misused.

The default behavior in most visualization tools is to assign a different color to each category. This treats color as a labeling system: each color means 'this is a different group.' But in data storytelling, color should mean something stronger: *this is the thing you should look at first.*

## The Two Uses of Color

**Color as category label** (use when you need to distinguish groups):
*Five continents get five colors so the reader can tell them apart.*

**Color as emphasis** (use when you have a story to tell):
*One country gets a bold color; everything else is gray so that country draws the eye immediately.*

The difference between these two uses is the difference between helping readers explore data and helping readers see your argument.

## Accessibility: Color and Vision

Approximately 8% of men and 0.5% of women have some form of color vision deficiency. The most common type (red-green color blindness) means that red and green look nearly identical.

**Practical rules:**
- Never use red and green to distinguish two categories that need to be compared
- Use blue and orange as a high-contrast, colorblind-friendly alternative
- Add shape or pattern as a second visual encoding when color alone is critical
- Test your charts with a colorblind simulation tool before publishing

## The Gray Principle

If everything is colorful, nothing is emphasized. Gray is underused in data visualization. When everything that is not part of your story is gray, the reader's eye goes immediately to what you want them to see.

---

<a name="read-code-7"></a>

**Cell 7 — Color as Emphasis vs. Color as Decoration**

This cell shows three versions of the same chart: one with every category a different color (decoration), one with a single emphasis color and gray for everything else (storytelling), and one demonstrating the red-green accessibility problem with the corrected blue-orange alternative.

<nav aria-label="Code cell 7 navigation">
<a href="#skip-code-7" aria-label="Skip code cell 7 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('white')

cont_life = gapminder[gapminder.year == 2007].groupby('continent')['lifeExp'].mean().reset_index()
cont_life = cont_life.sort_values('lifeExp', ascending=True)

# ── Version 1: Rainbow (decoration) ──────────────────────────────────────────
ax = axes[0]
colors_rainbow = ['#e74c3c','#e67e22','#f1c40f','#27ae60','#3498db']
bars = ax.barh(cont_life['continent'], cont_life['lifeExp'],
               color=colors_rainbow, edgecolor='white')
ax.set_title('❌ Rainbow Colors\n"Which continent should I look at?"',
             fontsize=11, color='#c0392b', pad=8)
ax.spines[['top','right']].set_visible(False)
ax.set_xlabel('Avg Life Expectancy (years)', fontsize=9)
ax.grid(axis='x', alpha=0.2)
ax.tick_params(labelsize=9)

# ── Version 2: Gray + emphasis (storytelling) ─────────────────────────────────
ax = axes[1]
story_colors = ['#cccccc' if c != 'Africa' else '#e67e22' for c in cont_life['continent']]
bars = ax.barh(cont_life['continent'], cont_life['lifeExp'],
               color=story_colors, edgecolor='white')
for bar, val, cont in zip(bars, cont_life['lifeExp'], cont_life['continent']):
    if cont == 'Africa':
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f} yrs', va='center', fontsize=10,
                fontweight='bold', color='#e67e22')
ax.set_title('✓ Gray + Emphasis\n"Africa has the shortest life expectancy"',
             fontsize=11, color='#27ae60', pad=8)
ax.spines[['top','right']].set_visible(False)
ax.set_xlabel('Avg Life Expectancy (years)', fontsize=9)
ax.grid(axis='x', alpha=0.2)
ax.tick_params(labelsize=9)

# ── Version 3: Accessibility — red/green problem and fix ─────────────────────
ax = axes[2]
categories = ['Survived', 'Did not survive']
values_ok   = [61.5, 38.5]

# Red-green (problematic)
ax.bar([0, 0.5], values_ok, width=0.3,
       color=['#2ecc71','#e74c3c'], edgecolor='white', label='Red-Green')
ax.text(0, values_ok[0]+1, '61.5%', ha='center', fontsize=10, color='#2ecc71')
ax.text(0.5, values_ok[1]+1, '38.5%', ha='center', fontsize=10, color='#e74c3c')
ax.text(0.25, -7, '❌ Red vs Green\n(invisible to 8% of men)',
        ha='center', fontsize=9, color='#c0392b')

# Blue-orange (accessible)
ax.bar([1.1, 1.6], values_ok, width=0.3,
       color=['#2980b9','#e67e22'], edgecolor='white', label='Blue-Orange')
ax.text(1.1, values_ok[0]+1, '61.5%', ha='center', fontsize=10, color='#2980b9')
ax.text(1.6, values_ok[1]+1, '38.5%', ha='center', fontsize=10, color='#e67e22')
ax.text(1.35, -7, '✓ Blue vs Orange\n(accessible to all)',
        ha='center', fontsize=9, color='#27ae60')

ax.set_xticks([0, 0.5, 1.1, 1.6])
ax.set_xticklabels(['Surv.','Not','Surv.','Not'], fontsize=8)
ax.set_title('Accessibility: Red-Green Problem\nand the Blue-Orange Fix',
             fontsize=11, fontweight='bold', pad=8)
ax.set_ylim(-12, 75)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig('color_principles.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-7"></a>

**In the gray-plus-emphasis version, the eye goes directly to Africa** — before the reader has read a single word. That is the goal: the visual should do the work of directing attention, so the narrative only needs to do the work of explaining why it matters.

On accessibility: if 8% of your male audience cannot distinguish your two main colors, your chart has a communication failure for 8% of your audience. Blue and orange are colorblind-safe, high-contrast, and widely accepted as a professional alternative to red and green.

<nav aria-label="Return navigation for code cell 7">
<a href="#read-code-7" aria-label="Go back and read code cell 7">&#8617; Go back and read the code cell</a>
</nav>

---
# Principle 7: Annotate the Insight

Most charts tell the reader where to look. Fewer charts tell the reader what to see when they get there.

**Annotation** is the practice of writing the story directly on the chart — not in a caption below, not in a paragraph beside it, but on the visualization itself, at the point where the insight lives.

A well-annotated chart is self-contained. The reader does not need to read surrounding text to understand it. They can pick it up cold and understand the argument in 10 seconds.

## What Good Annotations Do

- **Name what is being pointed at** — not just an arrow, but a label that explains the significance
- **Provide context directly** — *'+20 years since 1960'* rather than a data point floating in space
- **Answer the 'so what'** — *'This gap exceeds the average lifespan difference between smokers and non-smokers'*
- **Mark events that explain patterns** — an annotation that says *'2003: HIV antiretroviral treatment expanded'* turns a visible kink in a line from a mystery into a story

## Annotation as Narrative

The animated bubble chart Hans Rosling presented in 2006 was not a technical innovation. The data had been publicly available for decades. What Rosling added was narration — he told the story of each movement as it happened. He annotated every unexpected jump with an explanation: *'China — one child policy, dramatic fall in fertility.'* The animation was the visual. His voice was the annotation.

In a written data story, your annotation text does what his voice did.

---

<a name="read-code-8"></a>

**Cell 8 — The Fully Annotated Story Chart**

This cell applies all seven principles simultaneously to produce one chart that tells a complete, self-contained data story about Africa's development trajectory. Every element has a job. Anything without a job was removed.

<nav aria-label="Code cell 8 navigation">
<a href="#skip-code-8" aria-label="Skip code cell 8 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# All continents — gray context
for cont in gapminder.continent.unique():
    data = gapminder[gapminder.continent == cont].groupby('year')['lifeExp'].mean()
    ax.plot(data.index, data.values,
            color='#e0e0e0', linewidth=1.5, alpha=0.8, zorder=1)
    # Label at end
    if cont != 'Africa':
        ax.text(2008, data[2007], cont, fontsize=8, color='#aaaaaa', va='center')

# Africa — the story
africa = gapminder[gapminder.continent == 'Africa'].groupby('year')['lifeExp'].mean()
ax.plot(africa.index, africa.values,
        color='#e67e22', linewidth=3, zorder=5, marker='o', markersize=5)

# ── Annotations that tell the story ──────────────────────────────────────────

# Start point
ax.annotate(
    f'1952: {africa[1952]:.1f} years\n(starting point)',
    xy=(1952, africa[1952]),
    xytext=(1955, africa[1952] - 5),
    fontsize=9, color='#e67e22',
    arrowprops=dict(arrowstyle='->', color='#e67e22', lw=1.5)
)

# 1990s dip — HIV/AIDS crisis
ax.annotate(
    '1990s: HIV/AIDS crisis\nreverse gains in southern Africa',
    xy=(1997, africa[1997]),
    xytext=(1978, africa[1997] - 4),
    fontsize=9, color='#7f8c8d',
    arrowprops=dict(arrowstyle='->', color='#aaaaaa', lw=1.2)
)

# End point
ax.annotate(
    f'2007: {africa[2007]:.1f} years\n+{africa[2007]-africa[1952]:.1f} years gained',
    xy=(2007, africa[2007]),
    xytext=(1995, africa[2007] + 3),
    fontsize=10, color='#e67e22', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#e67e22', lw=1.5)
)

# Gap annotation
y_europe_2007 = gapminder[gapminder.continent=='Europe'].groupby('year')['lifeExp'].mean()[2007]
ax.annotate('', xy=(2007, y_europe_2007), xytext=(2007, africa[2007]),
            arrowprops=dict(arrowstyle='<->', color='#7f8c8d', lw=1.5))
ax.text(2009, (y_europe_2007 + africa[2007])/2,
        f'{y_europe_2007 - africa[2007]:.0f}-year\ngap remains',
        fontsize=9, color='#7f8c8d', va='center')

# ── Final styling ─────────────────────────────────────────────────────────────
ax.set_title(
    'Africa Has Gained 20 Years of Life Expectancy Since 1952 —\n'
    'Despite the HIV/AIDS Crisis Reversing a Decade of Progress',
    fontsize=13, fontweight='bold', pad=12, color='#2c3e50'
)
ax.set_ylabel('Average Life Expectancy (years)', fontsize=10, color='#666666')
ax.set_xlim(1950, 2018)
ax.set_ylim(35, 85)
ax.spines[['top','right']].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.grid(axis='y', color='#f0f0f0', linewidth=0.8)
ax.tick_params(colors='#666666', labelsize=9)

# Source note
ax.text(0.01, -0.08, 'Source: Gapminder Foundation. Values are continent averages.',
        transform=ax.transAxes, fontsize=8, color='#aaaaaa')

plt.tight_layout()
plt.savefig('annotated_story_chart.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-8"></a>

**This chart applies all seven principles:**

1. **Three pillars** — data (Gapminder), narrative (the HIV dip, the overall gain), visual (the chart)
2. **Story first** — the 'so what' is in the title: Africa gained 20 years despite a crisis
3. **Reader in mind** — the annotation about the HIV crisis provides context a general audience needs
4. **One claim per chart** — Africa's trajectory. Other continents are context, not the argument
5. **Decluttered** — no border, no legend, no gridlines on the x-axis
6. **Color as emphasis** — Africa is orange; everything else is gray
7. **Annotated** — the 1952 start, the 1990s dip, the 2007 endpoint, and the remaining gap are all labeled

A reader who has never seen this data can pick up this chart and understand the entire argument in 15 seconds. That is the goal.

<nav aria-label="Return navigation for code cell 8">
<a href="#read-code-8" aria-label="Go back and read code cell 8">&#8617; Go back and read the code cell</a>
</nav>

---
# The Master Class: Hans Rosling's Animated Chart

In 2006, Hans Rosling presented a TED talk called *'The best stats you've ever seen.'* It has been watched over 15 million times. He did not reveal new data. He did not use a proprietary dataset. Every number in his presentation was publicly available from the United Nations.

What he added was story.

He asked the question no one had asked visually: *What if you showed every country's health and wealth simultaneously, over 55 years, moving in real time?* And then he narrated what was happening as the viewer watched — explaining each unexpected movement, each reversal, each acceleration.

The result was not a data presentation. It was a worldview correction. He showed audiences that the mental model dividing the world into 'developed' and 'developing' was 50 years out of date.

The chart below recreates it. Watch what happens when you press play.

---

<a name="read-code-9"></a>

**Cell 9 — The Hans Rosling Animated Bubble Chart**

This cell produces the animated version of Rosling's famous visualization: GDP per capita on the x-axis, life expectancy on the y-axis, bubble size proportional to population, color by continent, animated year by year from 1952 to 2007. Press Play when the chart appears.

<nav aria-label="Code cell 9 navigation">
<a href="#skip-code-9" aria-label="Skip code cell 9 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
fig = px.scatter(
    gapminder,
    x='gdpPercap', y='lifeExp',
    animation_frame='year',
    animation_group='country',
    size='pop',
    color='continent',
    hover_name='country',
    log_x=True,
    size_max=60,
    range_x=[200, 120000],
    range_y=[25, 90],
    color_discrete_map={
        'Africa':   '#e74c3c',
        'Americas': '#3498db',
        'Asia':     '#f39c12',
        'Europe':   '#27ae60',
        'Oceania':  '#9b59b6'
    },
    labels={
        'gdpPercap': 'GDP per Capita (USD, log scale)',
        'lifeExp': 'Life Expectancy (years)',
        'pop': 'Population',
        'continent': 'Continent'
    },
    title='Health vs Wealth of Nations, 1952–2007  (bubble size = population)'
)

fig.update_layout(
    height=560,
    paper_bgcolor='white',
    plot_bgcolor='white',
    font_family='Arial',
    font_size=12,
    title_font_size=15,
    legend=dict(title='Continent', yanchor='top', y=0.99, xanchor='left', x=0.01)
)
fig.update_xaxes(
    gridcolor='#f0f0f0', linecolor='#cccccc',
    tickformat='$,.0f'
)
fig.update_yaxes(gridcolor='#f0f0f0', linecolor='#cccccc')

# Add annotations for context
fig.add_annotation(
    x=np.log10(1200), y=75,
    text='Asia closes the gap<br>with the West by 2007',
    showarrow=False, font=dict(size=11, color='#f39c12')
)

fig.show()
print('Press Play to animate the chart from 1952 to 2007.')

<a name="skip-code-9"></a>

**What to watch for as the animation plays:**

- Most of Asia starts clustered with Africa (bottom-left: poor and short-lived) in 1952. By 2007 China and Southeast Asia have moved dramatically toward the European cluster.
- The African cluster moves upward (longer lives) but less rightward (less economic growth) than the rest of the world over the same period.
- The two 'worlds' that existed in 1952 — healthy/wealthy and sick/poor — have become far more of a continuum by 2007.

Rosling's argument: *'There is no longer a "developing world" and a "developed world." There is a spectrum, and most of humanity has moved up it.'*

This is what data-driven storytelling achieves at its best: a factual correction to a false mental model, delivered in a way that is impossible to forget.

**Accessibility note:** Add a Markdown cell below this one describing the animation for someone who cannot see it — narrate the key movements from 1952 to 2007 as if you were Hans Rosling giving his talk.

<nav aria-label="Return navigation for code cell 9">
<a href="#read-code-9" aria-label="Go back and read code cell 9">&#8617; Go back and read the code cell</a>
</nav>

---
# Team Data Story
## Applying All Seven Principles to Your Final Project

Everything you have read and practiced in this notebook now applies to your own work.

The eight sections below are your deliverable. Each section applies one principle from the notebook to your team's final project data. Complete all eight. When you are finished, this section of the notebook should tell your data story clearly enough that someone who has never seen your project can understand what you found and why it matters.

**Requirements:**

- Every visualization must meet the accessibility standards taught in this course (4.5:1 color contrast, alt-text description in Markdown, descriptive chart titles)
- Every code cell must be run and show output — no error messages, no empty cells
- Every Markdown cell must contain your team's actual written answers — not placeholders, not the template text
- The story must be readable by someone outside your team without additional context

---

## Section 1: The Hook

**The goal:** Give your reader one surprising, specific, concrete fact in the first 30 seconds. Not your project title. Not your methodology. The one number or finding that makes someone lean forward and think: *'Wait — really?'*

Great hooks are:
- Specific, not vague (*'87% of patients'* not *'most patients'*)
- Counterintuitive or unexpected
- Directly relevant to someone who has not yet decided whether to keep reading

**Example (from the Gapminder story):**
*'The average person born in Africa today lives 20 years longer than one born in 1952 — a gain equivalent to the entire difference between smokers and non-smokers.'*

**✏️ Your Hook:**

*Write one to two sentences. This is the first thing your reader sees.*

---
**YOUR HOOK HERE**

---

<a name="read-code-10"></a>

**Team Code Cell 1 — Hook Visualization**

Create one visualization that makes your hook visible. This is a single chart that immediately supports your opening claim. Apply Principles 4, 5, 6, and 7: one claim, decluttered, color as emphasis, annotated with the finding.

Your chart title should restate your hook as a factual claim, not a description.

<nav aria-label="Code cell 10 navigation">
<a href="#skip-code-10" aria-label="Skip code cell 10 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# YOUR CODE HERE

# Load your project dataset here
# df = pd.read_csv('your_data.csv')

# Create your hook visualization
# Apply: one claim, declutter, color as emphasis, annotate the insight

# Requirements:
# - Chart title states the finding, not what the chart shows
# - If using color, ensure 4.5:1 contrast ratio
# - Use direct labels where possible instead of legends
# - Add at least one annotation that explains 'why this matters'

<a name="skip-code-10"></a>

**Accessibility note:** Add a Markdown cell below this one describing your chart in plain language for a screen reader user — describe what the chart shows, what the trend or pattern is, and what conclusion a reader should draw.

<nav aria-label="Return navigation for code cell 10">
<a href="#read-code-10" aria-label="Go back and read code cell 10">&#8617; Go back and read the code cell</a>
</nav>

**✏️ Write the narrative for your hook visualization here.**

In three to four sentences: what does this chart show, why is the finding surprising or important, and how does it connect to the problem your project addresses?

---
**YOUR NARRATIVE HERE**

---

## Section 2: The Problem and the Context

**The goal:** Establish the world as it was before your project — the status quo, the gap, the problem that existed before you started.

A story without a problem is an announcement. The reader needs to understand what was wrong, missing, or unknown before they can understand why your work matters.

**Principle in use:** *Find the story first.* Your context visualization should answer: what does the data look like before your intervention? What pattern, gap, or anomaly motivated the project?

**✏️ Your Problem Statement (two to three sentences):**

---
**YOUR PROBLEM STATEMENT HERE**

---

<a name="read-code-11"></a>

**Team Code Cell 2 — Context/Problem Visualization**

Create one visualization that establishes the problem context. This might be a baseline distribution, a trend showing decline, a gap between groups, or an anomaly in the data. Apply all seven principles. Your chart title should make the problem visible.

<nav aria-label="Code cell 11 navigation">
<a href="#skip-code-11" aria-label="Skip code cell 11 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# YOUR CODE HERE

# Context visualization
# This chart should show: what did the data look like before / without your solution?
# What is the gap, trend, anomaly, or disparity that motivated your project?

<a name="skip-code-11"></a>

**Accessibility note:** Add a Markdown cell below describing this chart in plain language.

<nav aria-label="Return navigation for code cell 11">
<a href="#read-code-11" aria-label="Go back and read code cell 11">&#8617; Go back and read the code cell</a>
</nav>

**✏️ Narrative for the context visualization (two to three sentences):**

---
**YOUR NARRATIVE HERE**

---

## Section 3: The Evidence

**The goal:** Show what your data reveals. This is the analytical heart of your story — two or three visualizations that build the case for your key finding.

Each visualization in this section should:
- Make exactly one claim (Principle 4)
- Be fully annotated with the insight (Principle 7)
- Meet accessibility standards (Principle 6)

The three visualizations should build on each other — each one answering a question the previous one raised, or showing the same finding from a different angle to increase confidence.

**Think of this as your data making the argument, step by step.**

<a name="read-code-12"></a>

**Team Code Cell 3a — Evidence Visualization 1**

First supporting visualization. State your 'so what' for this chart before building it: what specific claim does this chart make?

<nav aria-label="Code cell 12 navigation">
<a href="#skip-code-12" aria-label="Skip code cell 12 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# YOUR CODE HERE

# Evidence Visualization 1
# 'So what' for this chart: [write it as a comment before coding]
# Answer: This chart shows...

<a name="skip-code-12"></a>

**Accessibility note:** Add a Markdown cell below describing this chart.

<nav aria-label="Return navigation for code cell 12">
<a href="#read-code-12" aria-label="Go back and read code cell 12">&#8617; Go back and read the code cell</a>
</nav>

**✏️ Narrative (two sentences — state the finding, then explain why it matters):**

---
**YOUR NARRATIVE HERE**

---

<a name="read-code-13"></a>

**Team Code Cell 3b — Evidence Visualization 2**

Second supporting visualization. This should advance the argument — not repeat it in a different format.

<nav aria-label="Code cell 13 navigation">
<a href="#skip-code-13" aria-label="Skip code cell 13 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# YOUR CODE HERE

# Evidence Visualization 2
# 'So what' for this chart:

<a name="skip-code-13"></a>

**Accessibility note:** Add a Markdown cell below describing this chart.

<nav aria-label="Return navigation for code cell 13">
<a href="#read-code-13" aria-label="Go back and read code cell 13">&#8617; Go back and read the code cell</a>
</nav>

**✏️ Narrative:**

---
**YOUR NARRATIVE HERE**

---

<a name="read-code-14"></a>

**Team Code Cell 3c — Evidence Visualization 3**

Third supporting visualization — optional but recommended. If you have a third visualization, it should either confirm the finding from a different angle or show what the finding implies.

<nav aria-label="Code cell 14 navigation">
<a href="#skip-code-14" aria-label="Skip code cell 14 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# YOUR CODE HERE (optional — remove this cell if you only need two)

# Evidence Visualization 3
# 'So what' for this chart:

<a name="skip-code-14"></a>

**Accessibility note:** Add a Markdown cell below describing this chart.

<nav aria-label="Return navigation for code cell 14">
<a href="#read-code-14" aria-label="Go back and read code cell 14">&#8617; Go back and read the code cell</a>
</nav>

**✏️ Narrative:**

---
**YOUR NARRATIVE HERE**

---

## Section 4: The One Number

**The goal:** Distill your entire finding into one number your reader will remember.

After seeing all your evidence, what is the single most important quantitative result? Not seven metrics. Not a confusion matrix. One number — with context.

Context means:
- A comparison (*'compared to the baseline of...'*)
- A magnitude reference (*'equivalent to...'*)
- A direction (*'an increase of / a reduction of...'*)

**Examples:**
- *'Our model reduces missed diagnoses by 31% — equivalent to catching an additional 1 in 3 at-risk patients.'*
- *'Churn prediction accuracy improved from 67% to 89%, recovering an estimated $2.1M in annual retention value.'*
- *'Delivery time prediction error fell from 4.2 hours to 1.1 hours — below the industry threshold of 2 hours for the first time in the company's history.'*

**✏️ Your One Number:**

Write it in one sentence. Include the comparison or context that makes it meaningful.

---
**YOUR ONE NUMBER HERE**

---

<a name="read-code-15"></a>

**Team Code Cell 4 — The One Number Visualization**

Create a visualization that makes your one number the undeniable visual focus. This might be a single large bold number in a carefully designed figure, a before/after bar chart with the difference annotated, or a gauge showing improvement toward a target. Simple is correct here. The number should fill the visual.

<nav aria-label="Code cell 15 navigation">
<a href="#skip-code-15" aria-label="Skip code cell 15 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# YOUR CODE HERE

# The One Number visualization
# This should be your single most important result, made visually dominant
# Consider: a before/after comparison, a single annotated metric, or a simple callout

# Example structure:
# your_metric = 0.89  # your key result
# baseline    = 0.67  # what you're comparing to

# fig, ax = plt.subplots(figsize=(8, 4))
# ax.barh(['Your model', 'Baseline'], [your_metric, baseline],
#         color=['#2980b9', '#cccccc'], height=0.4)
# ax.axvline(x=your_metric, color='#2980b9', linestyle='--', alpha=0.4)
# ax.text(your_metric + 0.01, 0, f'{your_metric:.0%}', ...)

<a name="skip-code-15"></a>

**Accessibility note:** Add a Markdown cell below describing this visualization. A reader using a screen reader should be able to understand the key result from your description alone.

<nav aria-label="Return navigation for code cell 15">
<a href="#read-code-15" aria-label="Go back and read code cell 15">&#8617; Go back and read the code cell</a>
</nav>

**✏️ Narrative for the one number (one to two sentences — lead with the number, then the implication):**

---
**YOUR NARRATIVE HERE**

---

## Section 5: Acknowledging the Limitations

**The goal:** Build trust by being honest about what your data cannot say.

Every data story that claims more than the data can support loses credibility with a sophisticated audience. An industry expert who spots an unacknowledged limitation will question every other finding in your story.

Counterintuitively, acknowledging limitations increases trust — it signals that you understand the data and are not overselling it.

**What to address:**
- The most significant caveat about your dataset (size, sampling, time period, geography)
- The finding most likely to be questioned or challenged
- What additional data would strengthen your conclusions

**✏️ Your Limitations (three to five sentences):**

Be specific. *'The dataset is small'* is not a useful limitation statement. *'Our 300-sample dataset may not generalize to patients over 75, who were underrepresented in the training data'* is.

---
**YOUR LIMITATIONS HERE**

---

## Section 6: What Comes Next

**The goal:** End with forward momentum, not just a conclusion.

A data story that ends with *'In conclusion, our model achieved X'* has done half the job. The other half is telling the reader what to do with this information.

This section has three parts:

**The Implication:** What does your finding mean — for the field, for the organization, for the people affected?

**The Next Question:** What does your work make visible that was not visible before? What question do your findings raise that is now worth investigating?

**The Ask (optional):** If you were presenting to someone who could resource your next step, what would you ask them for?

**✏️ Your Team's Responses:**

*Implication (one to two sentences):*

*The next question your work raises:*

*The ask (if applicable):*

---
**YOUR RESPONSES HERE**

---

## Section 7: The Full Story — A One-Paragraph Summary

**The goal:** Write the complete data story in one paragraph.

This paragraph should be readable in under 60 seconds and contain:
- Your hook (the surprising or important opening finding)
- The problem you addressed
- The method in one clause (not a paper — a clause)
- Your key finding (the one number)
- The implication

This paragraph is your elevator pitch. It is what you would say if someone read your notebook and asked: *'So what's the bottom line?'*

Write it for the industry expert in your seminar audience — someone who is intelligent, time-constrained, and does not need you to define machine learning.

**✏️ Your One-Paragraph Story:**

---
**YOUR PARAGRAPH HERE**

---

## Section 8: Team Reflection

**Each team member completes this section individually** in the space below their name. You may discuss as a group but the writing should be your own.

**Answer the following (three to five sentences each):**

1. Which of the seven storytelling principles was hardest to apply to your project data, and why?
2. How did thinking about your reader (Principle 3) change the choices you made in this notebook?
3. What would you do differently in the analysis phase of the project, now that you have tried to tell its story?

---

**[Team Member 1 Name]:**

*Response here.*

**[Team Member 2 Name]:**

*Response here.*

**[Team Member 3 Name]:**

*Response here.*

**[Team Member 4 Name]:**

*Response here.*

---

---
# Submission Checklist

Before changing the sharing settings and submitting, confirm all of the following:

**Completeness:**
- All eight sections of the Team Data Story are filled in — no placeholder text remains
- Every team member's individual reflection (Section 8) is written
- All code cells have been run and show output — no error messages

**Visualization quality:**
- Every chart has a title that states the finding, not a description of the chart
- Every chart has at least one annotation explaining the key insight
- Color contrast meets the 4.5:1 minimum ratio (test with your browser's developer tools or an online checker)
- Direct labels are used where possible instead of separate legends
- Every visualization has a Markdown cell below it describing it in plain language

**Story quality:**
- The narrative cells read as prose, not bullet points
- The one-paragraph summary (Section 7) can stand alone
- A reader who has never seen your project can understand the story without asking questions

**Submission:**
- The host changes sharing to **Anyone with the link can view**
- The host submits the shared link to Canvas
- Only one submission per team — the link from the host's notebook

---

## A Final Note

Hans Rosling spent 20 years collecting and analyzing global health data before giving that TED talk in 2006. The analysis was not the hard part — it was sitting in databases waiting to be used. The hard part was figuring out how to make other people see what he saw.

That is the work of data-driven storytelling. It is not decoration added after the analysis. It is what transforms analysis into understanding, understanding into decision, and decision into change.

Your project findings are worth communicating well. This notebook is how you practice doing that.